# Assignment 1: exploratory data analysis and baselines

**Group 43, CENG313** · Mehmet Emircan Küllücek, Aleksei Khlopkov

Competition: [PhysioNet – Digitization of ECG Images](https://www.kaggle.com/competitions/physionet-ecg-image-digitization). It is closed, so we score locally on training records whose true signals we have.

**Task.** Given a picture of a printed 12-lead ECG, recover the 12 voltage curves (mV over time) that were printed on it.

This notebook
1. looks at the records, the ground-truth signals and the nine kinds of images,
2. tests how the official metric reacts to typical errors,
3. checks the page layout that our digitizer depends on,
4. scores two baselines with the official metric: **all zeros** and a **classical computer-vision digitizer** (no learning).

All reusable code is in `src/` (data loading, the official metric, the CV baseline, the page template). The notebook only calls it.

**Setup.** In Colab, first add a secret named `GITHUB_TOKEN` (key icon in the left sidebar) with a GitHub token that can read this private repo. The cell below clones the repo and downloads our dataset subset from Google Drive (about 2.5 GB, first run only). Running locally, start Jupyter from the repo root.

In [ ]:
# In Colab: clone the private repo with the GITHUB_TOKEN secret, then fetch the data.
# Locally: start Jupyter from the repo root and this only fetches the data if missing.
import os, sys

BRANCH = "main"  # set to a feature branch to run that branch's code

if "google.colab" in sys.modules and not os.path.isdir("src"):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !git clone -q -b {BRANCH} https://{token}@github.com/mmilesmorales/GROUP43_CENG313.git
    del token
    %cd GROUP43_CENG313

sys.path.insert(0, "src")
from data import download_dataset
download_dataset()  # ~2.5 GB on the first run, skipped when data/ exists
print("data ready:", len(os.listdir("data/train")), "record folders")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from data import load_val_ids, load_record_gt, extract_lead_signal
from metric import record_linear_snr, competition_score_from_linear
from utils import einthoven_residual
from baseline_cv import run_validation
import layout

DATA = "data"
LEADS = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
IMAGE_TYPES = ["0001", "0003", "0004", "0005", "0006", "0009", "0010", "0011", "0012"]
REC = 1006427285  # example record shown in the plots

train = pd.read_csv(f"{DATA}/train.csv")
fs_of = dict(zip(train.id, train.fs))
val_ids = load_val_ids()
eval_ids = val_ids[:30]  # fixed evaluation subset for all scores in this notebook
gt_ids = [r for r in val_ids if os.path.exists(f"{DATA}/train/{r}/{r}.csv")]
print(f"train.csv: {len(train)} records | validation split: {len(val_ids)} | "
      f"with ground truth in our subset: {len(gt_ids)} | evaluation subset: {len(eval_ids)}")

## 1. Records and sampling rates

`train.csv` has one row per record: `id`, sampling rate `fs` and signal length `sig_len`. Each record folder holds the ground-truth CSV (12 leads in mV) and nine images of the same ECG. `test.csv` lists, for every test image and lead, how many rows the prediction must have.

We split the records 90/10 **by record**, so all nine images of one ECG stay on the same side (`src/data.py`, seed 0). Our local subset contains the validation records; scores below use the first 30 of them.

In [ ]:
ax = train.fs.value_counts().sort_index().plot.bar(figsize=(6, 2.5), rot=0, color="tab:blue")
ax.set(xlabel="sampling rate fs (Hz)", ylabel="records", title="Sampling rates in train.csv")
plt.show()
print("every record is 10 s long (sig_len == 10 * fs):", bool((train.sig_len == 10 * train.fs).all()))
pd.read_csv(f"{DATA}/test.csv").head(12)

The six sampling rates are equally common (about 163 records each), and every record is 10 s long. So a prediction has to be resampled to a different number of rows per record; the rate is given, it never has to be guessed. Lead II needs four times as many rows as the other leads (next section explains why).

## 2. Ground-truth signals

The ground-truth CSV has `sig_len` rows for all 12 leads, but most cells are empty. A standard printout shows each lead for only 2.5 s: three rows of four leads, plus lead II again over the full 10 s at the bottom. The CSV mirrors this, so each lead is filled only in its own quarter of the 10 s. Below we draw one record the way it is printed.

In [ ]:
gt = load_record_gt(DATA, REC)
fs = fs_of[REC]
t = np.arange(len(gt)) / fs

fig, axes = plt.subplots(4, 1, figsize=(12, 7), sharex=True)
for i, (ax, row) in enumerate(zip(axes, layout.ROWS + (("II",),))):
    for lead in row:
        s = gt[lead].where(t < 2.5) if (lead == "II" and i < 3) else gt[lead]
        ax.plot(t, s, lw=0.7, color="k")
        start = t[s.notna().values][0]
        ax.text(start + 0.05, 0.9, lead, transform=ax.get_xaxis_transform(), fontsize=9, color="tab:red")
    ax.set_ylabel("mV")
    for q in (2.5, 5, 7.5):
        ax.axvline(q, color="0.8", lw=0.8)
axes[-1].set_xlabel("time (s)")
fig.suptitle(f"Ground truth of record {REC} ({fs} Hz), arranged like the printed page")
plt.tight_layout(); plt.show()

Next, statistics over all records with ground truth in our subset: how many samples each lead has, how big its amplitude is, and how much signal energy it carries.

In [ ]:
rows, einthoven = [], []
for r in gt_ids:
    g = load_record_gt(DATA, r)
    einthoven.append(einthoven_residual(g))
    for lead in LEADS:
        s = extract_lead_signal(g, lead)
        rows.append({"record": r, "fs": fs_of[r], "lead": lead, "n": len(s),
                     "peak_to_peak_mV": np.ptp(s), "energy": np.sum((s - s.mean()) ** 2)})
leads = pd.DataFrame(rows)

# Does floor(fs * 2.5) give the right number of rows for a short lead?
leads["floor_rule"] = np.floor(leads.fs * np.where(leads.lead == "II", 10, 2.5)).astype(int)
wrong = leads[leads.n != leads.floor_rule]
print(f"{len(gt_ids)} records, {len(leads)} lead series; floor(fs*2.5) is wrong for {len(wrong)} of them")
wrong.groupby(["fs", "lead"]).agg(true_rows=("n", "first"), floor_rule=("floor_rule", "first")).T

At 1025 Hz a quarter is 2562.5 samples, so the leads in quarters 2 and 4 get 2563 rows and those in quarters 1 and 3 get 2562. The formula `floor(fs * 2.5)` is wrong for half of the short leads at this rate. Row counts must match exactly for a submission, so we always take lengths from the ground truth or from `number_of_rows` in `test.csv`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
ax[0].boxplot([leads.loc[leads.lead == l, "peak_to_peak_mV"] for l in LEADS], showfliers=False)
ax[0].set_xticks(range(1, 13), LEADS)
ax[0].set(title="Peak-to-peak amplitude per lead", ylabel="mV")
share = leads.groupby("lead").energy.sum().reindex(LEADS) / leads.energy.sum() * 100
share.plot.bar(ax=ax[1], rot=0, color="tab:blue")
ax[1].set(title="Share of total signal energy", ylabel="%")
plt.tight_layout(); plt.show()
print("energy share of II + V2 + V3 + V4:", round(share[["II", "V2", "V3", "V4"]].sum(), 1), "%")

Typical beats are 1–2 mV tall, i.e. 80–160 px on the clean page. The chest leads V2–V4 are the largest, the augmented limb leads aVR/aVL/aVF the smallest.

The energy share matters because the metric adds signal and noise over all 12 leads of an image before dividing (section 4). Lead II (long strip) and V2–V4 together hold about 60% of the energy. Getting these four right decides most of the score; an error on aVL costs much less.

In [ ]:
e = pd.DataFrame(einthoven)
print("RMS of lead II:                 ", f"{e.rms_II.median() * 1000:7.1f} µV (median)")
print("RMS of II - (I + III):          ", f"{e.rms_resid_III.median() * 1000:7.2f} µV (median)")
print("RMS of aVR + aVL + aVF:         ", f"{e.rms_augmented_sum.median() * 1000:7.2f} µV (median)")

The six limb leads are not independent. By Einthoven's law II = I + III, and the augmented leads sum to zero. In the ground truth both hold to about 1 µV, which is only rounding. So a digitizer gets two free consistency checks, and a badly read limb lead can be repaired from the other two. We plan to use this in Assignment 2.

## 3. The nine image types

Every record comes as nine images of the same ECG. Type `0001` is a clean digital render; the others are the same page printed and then scanned or photographed, some with stains or damage. Below are the nine images of our example record.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for ax, typ in zip(axes.ravel(), IMAGE_TYPES):
    im = Image.open(f"{DATA}/train/{REC}/{REC}-{typ}.png")
    w, h = im.size
    im.thumbnail((1000, 1000))
    ax.imshow(im); ax.set_title(f"{typ}  ({w}×{h})", fontsize=10); ax.axis("off")
fig.suptitle(f"Record {REC}: the same ECG as nine image types", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
img_rows = []
for r in eval_ids:
    for typ in IMAGE_TYPES:
        p = f"{DATA}/train/{r}/{r}-{typ}.png"
        if os.path.exists(p):
            with Image.open(p) as im:  # reads the header only
                img_rows.append({"type": typ, "width": im.width, "height": im.height,
                                 "mode": im.mode, "MB": os.path.getsize(p) / 1e6})
imgs = pd.DataFrame(img_rows)
imgs.groupby("type").agg(images=("width", "size"), width=("width", "median"), height=("height", "median"),
                         portrait=("height", lambda h: (h > imgs.loc[h.index, "width"]).mean()),
                         mode=("mode", lambda m: m.mode()[0]), median_MB=("MB", "median")).round(2)

What we see (from the example record and the size table):

| type | what it is | why it is hard |
|---|---|---|
| 0001 | clean render, 2200×1700, red grid | nothing; the exact page template |
| 0003, 0011 | colour scans (0011 with stains) | slightly smaller and shifted; faded colours |
| 0004, 0012 | black-and-white scans | the grid is as dark as the trace, so colour can't separate them |
| 0005, 0010 | phone photos of the printout, ~12 MP | perspective, curved paper, background around the page |
| 0006 | photo of the page on a screen | moiré, glare, dark bezel |
| 0009 | phone photo, rotated 90°, stained paper | rotation, stains, uneven light |

Only `0001` matches the page template pixel for pixel. Every other type first has to be mapped back onto it (rotation, perspective, scale). This is why the classical baseline below works on clean pages and fails on most others.

## 4. How the metric reacts to errors

The official metric (vendored in `src/metric.py`) scores each image as follows:
1. per lead, shift the prediction in time by the best lag within ±0.2 s and remove the best constant offset;
2. add up signal power (truth²) and noise power ((truth − prediction)²) over all 12 leads;
3. SNR of the image = signal / noise. The final score is 10·log10 of the **mean SNR over images**; a perfect image is capped at 60 dB.

To see what this rewards, we damage the true signals of the 30 evaluation records on purpose and score them.

In [ ]:
eval_gt = {r: load_record_gt(DATA, r) for r in eval_ids}

def score_of(transform):
    """Score the ground truth of the evaluation records after applying transform(signal, fs) to every lead."""
    ratios = []
    for r, g in eval_gt.items():
        fs = fs_of[r]
        pred = {lead: transform(extract_lead_signal(g, lead), fs) for lead in LEADS}
        ratios.append(record_linear_snr(g, pred, fs))
    return competition_score_from_linear(ratios)

def delayed(s, k):
    return np.concatenate([np.full(k, s[0]), s[:-k]])

def stretched(s, e):  # read the time axis e (e.g. 1%) too slow
    return np.interp(np.arange(len(s)) * (1 + e), np.arange(len(s)), s)

rng = np.random.default_rng(0)
tests = {
    "all zeros": lambda s, fs: np.zeros_like(s),
    "+0.5 mV offset": lambda s, fs: s + 0.5,
    "delay 0.1 s": lambda s, fs: delayed(s, int(0.1 * fs)),
    "delay 0.3 s": lambda s, fs: delayed(s, int(0.3 * fs)),
    "noise, sd 0.02 mV": lambda s, fs: s + rng.normal(0, 0.02, len(s)),
    "noise, sd 0.05 mV": lambda s, fs: s + rng.normal(0, 0.05, len(s)),
}
for e in (0.02, 0.05, 0.1, 0.2):
    tests[f"amplitude {e:.0%} too large"] = lambda s, fs, e=e: s * (1 + e)
for e in (0.005, 0.01, 0.02):
    tests[f"time axis {e:.1%} stretched"] = lambda s, fs, e=e: stretched(s, e)

metric_tests = pd.Series({name: score_of(f) for name, f in tests.items()}, name="score (dB)").round(2)
metric_tests.to_frame()

In [ ]:
err = np.array([0.5, 1, 2, 5, 10, 20])
gain_db = [score_of(lambda s, fs, e=e: s * (1 + e / 100)) for e in err]
time_db = [score_of(lambda s, fs, e=e: stretched(s, e / 100)) for e in err]
plt.figure(figsize=(6, 3.5))
plt.semilogx(err, gain_db, "o-", label="amplitude scale error")
plt.semilogx(err, time_db, "s-", label="time scale error")
plt.axhline(0, color="0.5", lw=0.8)
plt.xlabel("error (%)"); plt.ylabel("score (dB)"); plt.legend()
plt.title("Same-size errors, very different cost"); plt.grid(alpha=0.3); plt.show()

Lessons for building a digitizer:
- **Vertical offset is free.** We don't need to find the exact 0 mV line, only the right shape and scale.
- **Time shifts up to 0.2 s are found by the metric.** A 0.1 s delay still scores about 21 dB; the loss comes only from the 0.1 s at the start of each lead that is missing. A 0.3 s delay is outside the search window and scores below zero.
- **Amplitude scale errors are moderately expensive.** An error e caps the SNR near 1/e²: 5% → about 26 dB, 10% → about 20 dB.
- **Time-scale errors are the most expensive.** Stretching the time axis by 1% loses more than a 10% amplitude error, because the beats drift out of phase along the strip. Getting px/s right matters more than px/mV.
- **All zeros scores ≈ 0 dB**, not minus infinity. Offset removal lets a flat line explain a little of the signal. So for a hopeless image, predicting zeros beats a wrong guess.

## 5. Checking the page template

The baseline and our later work assume a fixed layout of the clean `0001` page (`src/layout.py`): 200 dpi, 25 mm/s and 10 mm/mV, i.e. **196.85 px/s and 78.74 px/mV**, signals starting at x = 118 px, and the 0 mV lines of the four rows at y = 709, 992, 1275 and 1535 px.

We check this directly. We draw the **true** signal at the template position and count how many samples land on dark ink (±2 px). As a control, we draw the ground truth of a *different* record on the same image. If the template is right, the own record should hit almost everywhere and the other record much less.

In [ ]:
img = np.asarray(Image.open(f"{DATA}/train/{REC}/{REC}-0001.png").convert("RGB"))
gt, fs = load_record_gt(DATA, REC), fs_of[REC]

fig, ax = plt.subplots(figsize=(13, 6))
ax.imshow(img)
for r, row in enumerate(layout.ROWS):
    for c, lead in enumerate(row):
        s = extract_lead_signal(gt, lead)[: int(round(2.5 * fs))]
        ax.plot(*layout.lead_positions(s, fs, r, c), lw=0.6, color="tab:blue", alpha=0.8)
ax.plot(*layout.lead_positions(extract_lead_signal(gt, "II"), fs, 3, 0), lw=0.6, color="tab:blue", alpha=0.8)
ax.set(xlim=(100, 1250), ylim=(1650, 600), title="Ground truth (blue) drawn on the clean image at the template position")
ax.axis("off"); plt.show()

In [ ]:
def hit_rates(img_id, gt_id):
    ink = layout.ink_mask(np.asarray(Image.open(f"{DATA}/train/{img_id}/{img_id}-0001.png").convert("RGB")))
    g, fs = load_record_gt(DATA, gt_id), fs_of[gt_id]
    out = {}
    for r, row in enumerate(layout.ROWS):
        for c, lead in enumerate(row):
            s = extract_lead_signal(g, lead)[: int(round(2.5 * fs))]
            out[lead] = layout.ink_hit_rate(ink, *layout.lead_positions(s, fs, r, c))
    out["II (10 s)"] = layout.ink_hit_rate(ink, *layout.lead_positions(extract_lead_signal(g, "II"), fs, 3, 0))
    return out

own = pd.DataFrame([hit_rates(r, r) for r in eval_ids])
other = pd.DataFrame([hit_rates(r, eval_ids[(i + 1) % len(eval_ids)]) for i, r in enumerate(eval_ids)])
pd.DataFrame({"own record, mean": own.mean(), "own record, worst": own.min(),
              "other record, mean": other.mean()}).mul(100).round(1).T

On all 30 clean images, practically every true sample of every lead lands on ink at the template position, while the control (another record's signal) hits far less. So the template is right, for every lead including the 10 s strip. On type `0001` the whole problem is reading the trace; the geometry is known.

## 5. Baseline 1 — all zeros (~0 dB anchor)

In [ ]:
val_ids = load_val_ids(30)      # fixed 30-record subset (record-level 90/10 split, seed 0)
have = [r for r in val_ids if os.path.exists(f"{DATA}/train/{r}/{r}.csv")]
ratios = []
for rec in have:
    g = load_record_gt(DATA, rec)
    ratios.append(record_linear_snr(g, {L: np.zeros(len(g[L].dropna())) for L in g.columns}, fs_of[str(rec)]))
print(f"records scored: {len(have)}")
print(f"BASELINE 1 (all zeros): {competition_score_from_linear(ratios):.3f} dB")

## 6. Baseline 2 — classical CV digitization (no ML)
`src/baseline_cv.py`: grid autocorrelation → calibration (px/s, px/mV) → fixed-layout
cell extraction → per-column trace tracking → resample → official metric. Evaluated on
the same 30-record subset across **all 9 variants** (270 evaluations).

In [ ]:
have_pngs = [r for r in val_ids if all(os.path.exists(f"{DATA}/train/{r}/{r}-{v}.png") for v in VARIANTS)]
print("records with full image set:", len(have_pngs))
df_cv = run_validation(have_pngs, out_csv="cv_scores.csv", verbose=False)
print(df_cv.groupby('variant').db.agg(['mean','min','max']).round(2).to_string())
print(f"\nBASELINE 2 (classical CV): {competition_score_from_linear(df_cv.ratio):.3f} dB")

## 7. Results summary

In [ ]:
summary = {
    "all-zeros": competition_score_from_linear(ratios),
    "classical CV": competition_score_from_linear(df_cv.ratio),
    "community pretrained (same cohort, reference)": 17.989,
}
print(f"{'baseline':46s} {'score (dB)':>10s}")
for k, v in summary.items(): print(f"{k:46s} {v:10.3f}")
print("\nreference only — competition winner 23.33 dB on the PRIVATE test set (different data)")

**Takeaways**
- Pure-geometry CV is *correct on clean scans* (up to +15 dB/record) but collapses on
real-world variants (168/270 records below the zeros floor) — quantifying exactly why
successful approaches begin with learned rectification.
- The all-zeros anchor sits at ≈ 0 dB: the metric's vertical-offset removal gives small
positive boosts (≤ +3.9 dB/record).
- Everything is scored with the official metric on a fixed record-level validation split.